# Phase 5 — Market Basket Analysis
## Retail Margin Dynamics & Discount Optimization

This notebook demonstrates transaction co-occurrence mining using the **Apriori** algorithm from `mlxtend`.

> **Caution:** Association rules identify empirical co-purchase patterns in historical transactions. Association does **not** imply causation.

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Load cleaned data
df = pd.read_csv('../data/processed/superstore_cleaned.csv', low_memory=False)
print(f'Total line items: {len(df):,}')
print(f'Distinct orders: {df["order_id"].nunique():,}')

In [ ]:
# 2. Analyze basket sizes
order_sizes = df.groupby('order_id')['product_name'].nunique()
multi_orders = order_sizes[order_sizes >= 2]
print(f'Multi-item orders (effective basket pool): {len(multi_orders):,} ({len(multi_orders)/df["order_id"].nunique()*100:.1f}%)')

In [ ]:
# 3. Filter candidate items and construct basket matrix
top_prods = set(df['product_name'].value_counts()[lambda x: x >= 25].index)
baskets = (
    df[df['product_name'].isin(top_prods)]
    .groupby('order_id')['product_name']
    .apply(lambda s: list(set(s)))
    .tolist()
)
baskets = [b for b in baskets if len(b) >= 2]

te = TransactionEncoder()
te_ary = te.fit(baskets).transform(baskets)
df_basket = pd.DataFrame(te_ary, columns=te.columns_)
print(f'Basket matrix shape: {df_basket.shape}')

In [ ]:
# 4. Apriori Frequent Itemsets & Rules
min_support = 2 / len(baskets)
frequent = apriori(df_basket, min_support=min_support, use_colnames=True, max_len=3)
rules = association_rules(frequent, metric='lift', min_threshold=1.0)
filtered_rules = rules[(rules['confidence'] >= 0.10) & (rules['lift'] > 1.0)].sort_values(by=['lift', 'confidence'], ascending=[False, False])
print(f'Retained rules: {len(filtered_rules):,}')
filtered_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)